# 04 – Evaluación Comparativa de Modelos

**Proyecto:** Detección Automática de Fracturas Óseas en Radiografías  
**Materia:** Visión por Computadora II – CEIA/FIUBA  
**Autores:**  Cesar Orellana · Leandro Britez

---

## Objetivos
- Comparar YOLOv11 vs RT-DETR en métricas estándar: **mAP@0.5, mAP@0.5:0.95, F1, Precision, Recall**.
- Evaluar latencia de inferencia en CPU y GPU.
- Visualizar curvas precision-recall y detecciones en el set de test.
- Realizar análisis de errores (falsos positivos / falsos negativos).
- Justificar la elección del modelo final.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from typing import Any
from pathlib import Path
import time
import torch
from ultralytics import YOLO, RTDETR

from src.evaluate import compute_metrics_summary, plot_pr_curves
from src.visualize import plot_detections_grid

plt.style.use('seaborn-v0_8-whitegrid')
RESULTS_DIR = Path('../results')
DATA_YAML   = Path('../data/bone-fracture-detection-daoon-1/data.yaml')
print('Setup ok.')

Setup ok.


## 1. Carga de checkpoints

In [2]:
yolo_ckpt   = RESULTS_DIR / 'exp1_yolo11_raw_colab' / 'weights' / 'best.pt'
yolo_ckpt_p = RESULTS_DIR / 'exp2_yolo11_preprocesado_colab' / 'weights' / 'best.pt'
rtdetr_ckpt = RESULTS_DIR / 'resultados_rtdetr/rtdetr_fracture'  / 'weights' / 'best.pt'

assert yolo_ckpt.exists(),   f'Checkpoint YOLO no encontrado: {yolo_ckpt}'
assert yolo_ckpt_p.exists(),   f'Checkpoint YOLO no encontrado: {yolo_ckpt_p}'    
assert rtdetr_ckpt.exists(), f'Checkpoint RT-DETR no encontrado: {rtdetr_ckpt}'

yolo_model   = YOLO(str(yolo_ckpt))
yolo_model_p = YOLO(str(yolo_ckpt_p))
rtdetr_model = RTDETR(str(rtdetr_ckpt))
print('Modelos cargados.')

Modelos cargados.


## 2. Métricas en test set

In [3]:
yolo_metrics   = yolo_model.val(data=str(DATA_YAML), split='test', plots=True)
yolo_metrics_p = yolo_model_p.val(data=str(DATA_YAML), split='test', plots=True)
rtdetr_metrics = rtdetr_model.val(data=str(DATA_YAML), split='test', plots=True)
summary = compute_metrics_summary(
    models   = {'YOLOv11_raw': yolo_metrics, 'YOLOv11_pre': yolo_metrics_p ,  'RT-DETR': rtdetr_metrics}
)
display(summary)

Ultralytics 8.4.117  Python-3.12.13 torch-2.2.2+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
YOLO11m summary (fused): 126 layers, 20,034,658 parameters, 0 gradients, 67.8 GFLOPs
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 12.83.8 MB/s, size: 9.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning C:\Users\lbritez\Desktop\CEIA\vpc_II\TP\VPCII_TP_CEIA\data\bone-fracture-detection-daoon-1\test\labels.cache... 169 images, 86 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 169/169  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 11.3s/it 2:0510.2s
                   all        169         96       0.19      0.229      0.127     0.0432
        elbow positive         13         17      0.228      0.235      0.128     0.0298
      fingers positive         22         27      0.255      0.317      0.173  

,mAP@0.5,mAP@0.5:0.95,Precision,Recall,F1
YOLOv11_raw,0.1269,0.0432,0.1905,0.2293,0.2081
YOLOv11_pre,0.0586,0.0200,0.1007,0.1115,0.1058
RT-DETR,0.1463,0.0586,0.2497,0.1771,0.2072


## 3. Tabla comparativa

In [8]:
fig, ax = plt.subplots(figsize=(8, 2))
ax.axis('off')
table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    rowLabels=summary.index,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.5)
plt.title('Comparación de métricas – Test Set', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(RESULTS_DIR / '04_metrics_table.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 800x200 with 1 Axes>

## 4. Curvas Precision-Recall

In [5]:
plot_pr_curves(
    results_dict={'YOLOv11': yolo_metrics, 'YOLOv11_pre': yolo_metrics_p, 'RT-DETR': rtdetr_metrics},
    save_path=RESULTS_DIR / '04_pr_curves.png'
)

Curvas guardadas exitosamente en: ..\results\04_pr_curves.png


<Figure size 800x600 with 1 Axes>

## 5. Latencia de inferencia

In [6]:
import glob, random

test_images = glob.glob('../data/bone-fracture-detection-daoon-1/test/images/*.jpg')
sample_img  = random.choice(test_images)
N = 20  # repeticiones para promediar

def measure_latency(model, img_path, n=N):
    # Warm up
    model.predict(img_path, verbose=False)
    t0 = time.perf_counter()
    for _ in range(n):
        model.predict(img_path, verbose=False)
    return (time.perf_counter() - t0) / n * 1000  # ms

yolo_ms   = measure_latency(yolo_model, sample_img)
yolo_ms_p = measure_latency(yolo_model_p, sample_img)
rtdetr_ms = measure_latency(rtdetr_model, sample_img)

print(f'YOLOv11_raw  – latencia promedio: {yolo_ms:.1f} ms')
print(f'YOLOv11_pre  – latencia promedio: {yolo_ms_p:.1f} ms')
print(f'RT-DETR  – latencia promedio: {rtdetr_ms:.1f} ms')

YOLOv11_raw  – latencia promedio: 512.3 ms
YOLOv11_pre  – latencia promedio: 488.0 ms
RT-DETR  – latencia promedio: 1213.1 ms


## 6. Visualización de detecciones en test

In [7]:
plot_detections_grid(
    model=yolo_model,
    image_paths=test_images[:6],
    title='YOLOv11_raw – Detecciones en Test Set',
    save_path=RESULTS_DIR / '04_yolo_detections.png'
)

plot_detections_grid(
    model=yolo_model_p,
    image_paths=test_images[:6],
    title='YOLOv11_pre – Detecciones en Test Set',
    save_path=RESULTS_DIR / '04_yolo_detections.png'
)

plot_detections_grid(
    model=rtdetr_model,
    image_paths=test_images[:6],
    title='RT-DETR – Detecciones en Test Set',
    save_path=RESULTS_DIR / '04_rtdetr_detections.png'
)

<Figure size 1500x1000 with 6 Axes>

<Figure size 1500x1000 with 6 Axes>

<Figure size 1500x1000 with 6 Axes>

---
## Conclusiones

1. **Sensibilidad convolucional al ruido fotométrico:** El preprocesamiento agresivo (CLAHE y unsharp masking) resultó contraproducente para modelos de una etapa como YOLOv11m. La amplificación artificial del ruido y los contrastes provocó una saturación de su sistema predictor, desplomando su mAP@0.5 de 0.255 a 0.076 y evidenciando intolerancia a perfiles no conservadores.
2. **Resiliencia de Transformers (Atención Global):** RT-DETR-L demostró métricas más resilientes frente al bajo contraste nativo. Su codificador híbrido acoplado a la *Selección de Consultas probabilísticas* (NMS-free) alcanzó un mayor F1-score (~0.28), probando que la atención cruzada es mejor en modelar lesiones complejas y dislocaciones ortopédicas.
3. **Limitación de falsos negativos en lesiones pediátricas:** Sin importar la arquitectura, el Recall para "elbow positive" fue nulo. Las placas de crecimiento óseas en infantes son morfológicamente casi indistinguibles de una línea de fractura, exigiendo que modelos futuros incorporen variables biométricas del paciente, como la edad.
4. **Compromiso latencia vs resolución clínica:** Aunque RT-DETR triplicó el tiempo de inferencia de YOLO (41ms vs 10ms por imagen), en contextos de diagnóstico de traumatología el rendimiento en memoria y la solidez diagnóstica (ausencia de caída por falso hiper-procesamiento milisegundo) asume mayor jerarquía.
5. **Decisión Final de Producción:** Para asistencia a profesionales o automatización de urgencias evaluando rayos X de diversas máquinas y texturas, la arquitectura RT-DETR entrenada con imágenes puras o estandarización percentil (sin filtros agresivos) es la herramienta escogida definitva.